In [ ]:
!pip install fastapi uvicorn vosk python-multipart nest_asyncio pyngrok
!apt-get update && apt-get install -y ffmpeg

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 77.9 MB/s eta 0:00:00:00:0100:01
  Created wheel for srt: filename=srt-3.5.3-py3-none-any.whl size=22427 sha256=0cb4d29bea4418972edf831df0aba1e0802941b56a2e9abfc5fa522083942021
  Stored in directory: /root/.cache/pip/wheels/7e/75/5b/e1d5c3756631e4bda806f6cc9640153b39484bb6f7b0b8def3
Successfully built srt
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:3 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:7 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:8 https://r2u.stat.illinois.edu

In [2]:
import os
import urllib.request
import zipfile

model_dir = "vosk-model-ar-mgb2-0.4"

if not os.path.exists(model_dir):
    print("Downloading Arabic model (500MB)...")
    model_url = "https://alphacephei.com/vosk/models/vosk-model-ar-mgb2-0.4.zip"
    model_zip = "vosk-model-ar-mgb2-0.4.zip"
    
    urllib.request.urlretrieve(model_url, model_zip)
    print("Extracting...")
    
    with zipfile.ZipFile(model_zip, 'r') as zip_ref:
        zip_ref.extractall(".")
    
    os.remove(model_zip)
    print(f"Model ready at: {model_dir}")
else:
    print(f"Model already exists at: {model_dir}")

Extracting...
Model ready at: vosk-model-ar-mgb2-0.4


In [ ]:
import os
import json
import tempfile
import subprocess
from vosk import Model, KaldiRecognizer
import wave

# Model path for notebook
MODEL_PATH = "vosk-model-ar-mgb2-0.4"

# Load model once
_model = None

def get_model():
    """Lazy load the Vosk model"""
    global _model
    if _model is None:
        if not os.path.exists(MODEL_PATH):
            raise FileNotFoundError(f"Vosk model not found at {MODEL_PATH}")
        _model = Model(MODEL_PATH)
    return _model

def convert_to_wav(input_file: str) -> str:
    """Convert audio file to WAV format (16kHz, mono, 16-bit PCM)"""
    output_file = tempfile.mktemp(suffix=".wav")

    cmd = [
        'ffmpeg',
        '-i', input_file,
        '-ar', '16000',
        '-ac', '1',
        '-c:a', 'pcm_s16le',
        '-y',
        output_file
    ]

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        return output_file
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"FFmpeg conversion failed: {e.stderr}")
    except FileNotFoundError:
        raise RuntimeError("FFmpeg not installed")

async def transcribe_audio(audio_path: str) -> str:
    """Transcribe Arabic audio file using Vosk"""

    # Always convert the input audio to the required WAV format to ensure consistency
    # and handle potential issues with input WAV files not conforming to expected format.
    wav_file = convert_to_wav(audio_path)
    temp_file = wav_file # Mark for cleanup

    try:
        wf = wave.open(wav_file, "rb")

        # These checks are now redundant if convert_to_wav always produces the correct format,
        # but keeping them for an extra layer of validation or if convert_to_wav changes.
        if wf.getnchannels() != 1 or wf.getsampwidth() != 2:
            raise ValueError("Audio must be mono 16-bit PCM")

        model = get_model()
        rec = KaldiRecognizer(model, wf.getframerate())
        rec.SetWords(True)

        full_text = []

        while True:
            data = wf.readframes(4000)
            if len(data) == 0:
                break

            if rec.AcceptWaveform(data):
                result = json.loads(rec.Result())
                text = result.get("text", "").strip()
                if text:
                    full_text.append(text)

        final_result = json.loads(rec.FinalResult())
        final_text = final_result.get("text", "").strip()
        if final_text:
            full_text.append(final_text)

        wf.close()

        transcription = " ".join(full_text)
        return transcription if transcription else "لم يتم اكتشاف أي كلام"

    finally:
        if temp_file and os.path.exists(temp_file):
            os.remove(temp_file)

print("✅ Service functions loaded")

✅ Service functions loaded


In [4]:
from fastapi import FastAPI, APIRouter, File, UploadFile, HTTPException
from fastapi.responses import JSONResponse
import tempfile
import nest_asyncio

# Allow nested event loops (needed for notebooks)
nest_asyncio.apply()

# Create FastAPI app
app = FastAPI(title="STT API")
router = APIRouter()

@router.post("/convert")
async def convert_speech_to_text(
    audio: UploadFile = File(..., description="Audio file (WAV, MP3, M4A, etc.)")
):
    """Convert Arabic speech to text"""
    
    # Validate file size (10MB max)
    content = await audio.read()
    max_size = 10 * 1024 * 1024
    if len(content) > max_size:
        raise HTTPException(400, f"File too large. Max size: {max_size / (1024*1024)}MB")
    
    # Save to temp file
    temp_file = tempfile.mktemp(suffix=os.path.splitext(audio.filename)[1])
    
    try:
        with open(temp_file, "wb") as f:
            f.write(content)
        
        # Transcribe
        transcription = await transcribe_audio(temp_file)
        
        return JSONResponse({
            "success": True,
            "transcription": transcription,
            "filename": audio.filename
        })
        
    except Exception as e:
        raise HTTPException(500, f"Transcription failed: {str(e)}")
        
    finally:
        if os.path.exists(temp_file):
            os.remove(temp_file)

@router.get("/health")
async def health():
    return {"service": "stt", "status": "healthy", "model": "vosk-ar-mgb2-0.4"}

@app.get("/")
async def root():
    return {"message": "STT API", "version": "1.0.0"}

# Include router
app.include_router(router, prefix="/api/stt", tags=["STT"])

print("✅ FastAPI app created")

✅ FastAPI app created


In [14]:
import uvicorn
import os
from uvicorn.config import Config
from uvicorn.server import Server
import asyncio # Import asyncio
from pyngrok import ngrok # Import ngrok

# Set ngrok auth token
ngrok.set_auth_token("32kzkKWqRqOc6YXx3yo1QjFdqL8_qr8YTJFqnyVfcCsfL858")

port = 8001 # Keeping the port at 8001 to avoid 'address already in use' from previous attempts

# Start ngrok tunnel (optional - for public access)
try:
    public_url = ngrok.connect(port)
    print(f"\n{'='*60}")
    print(f"🚀 Public URL: {public_url}")
    print(f"{'-'*60}")
    print(f"Local URL: http://localhost:{port}") # Also show local URL
    print(f"{'='*60}\n")
except Exception as e:
    print(f"⚠️  Ngrok configuration failed: {e}. Running locally only")
    print(f"Local URL: http://localhost:{port}")

# Run server
print(f"\n📡 Starting server on port {port}...")
print(f"📄 API Docs: http://localhost:{port}/docs")
print(f"🔍 STT Endpoint: POST http://localhost:{port}/api/stt/convert\n")

# Explicitly configure and run the server to avoid `loop_factory` issues with nest_asyncio
config = Config(app, host="0.0.0.0", port=port, log_level="info")
server = Server(config)

# Run the server's serve coroutine directly using asyncio.run
# This works because nest_asyncio has already patched asyncio.run to allow nested event loops.
asyncio.run(server.serve())

                                                                                                    
🚀 Public URL: NgrokTunnel: "https://511af6ad9907.ngrok-free.app" -> "http://localhost:8001"
------------------------------------------------------------
Local URL: http://localhost:8001


📡 Starting server on port 8001...
📄 API Docs: http://localhost:8001/docs
🔍 STT Endpoint: POST http://localhost:8001/api/stt/convert



INFO:     Started server process [259]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


INFO:     197.43.217.103:0 - "GET / HTTP/1.1" 200 OK
INFO:     197.43.217.103:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     197.43.217.103:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     197.43.217.103:0 - "POST /api/stt/convert HTTP/1.1" 500 Internal Server Error
INFO:     197.43.217.103:0 - "POST /api/stt/convert HTTP/1.1" 200 OK
INFO:     197.43.217.103:0 - "POST /api/stt/convert HTTP/1.1" 200 OK
INFO:     197.43.217.103:0 - "POST /api/stt/convert HTTP/1.1" 500 Internal Server Error
INFO:     197.43.217.103:0 - "POST /api/stt/convert HTTP/1.1" 200 OK
INFO:     197.43.217.103:0 - "POST /api/stt/convert HTTP/1.1" 500 Internal Server Error


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [259]


KeyboardInterrupt: 